# Experiment: iWildCam CVaR Analysis (ERM, GroupDRO, IRO)

This notebook runs the unified CVaR collection pipeline and plots DGIL-style CVaR-vs-alpha curves using SciencePlots styling.


In [ ]:
from __future__ import annotations

import json
import os
import shlex
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

for candidate in [Path.cwd(), Path.cwd().parent, Path("/home/c01josh/CISPA-scratch/c01josh/iro")]:
    if (candidate / "iro").exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))
        break

from iro.visualization.cvar_curves import plot_cvar_alpha_curves


## Configuration

Set these paths to your cluster workspace.


In [ ]:
REPO_ROOT = Path("/home/c01josh/CISPA-scratch/c01josh/iro")
RESULTS_ROOT = REPO_ROOT / "runs/iwildcam_long/results"
DATA_ROOT = Path("/home/c01josh/CISPA-scratch/c01josh/datasets/iwildcam")

OUT_ROOT = REPO_ROOT / "collected_results/iwildcam_cvar"
EXPERIMENT = "iwildcam_iro"
SPLIT = "val"
ALGORITHMS = "erm,groupdro,iro"
ALPHA_GRID = "0.0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0"
DEVICE = "cuda"

assert REPO_ROOT.exists(), f"Missing repo root: {REPO_ROOT}"
assert DATA_ROOT.exists(), f"Missing dataset root: {DATA_ROOT}"
print("REPO_ROOT:", REPO_ROOT)
print("RESULTS_ROOT:", RESULTS_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("OUT_ROOT:", OUT_ROOT)


## Submit all three algorithms in one go (Slurm)

Run this command in a terminal to submit `erm`, `groupdro`, and `iro` together.


In [ ]:
submit_cmd = f"""
cd {REPO_ROOT} && \
DRY_RUN=0 \
ARRAY_RANGE=0-4 \
IRO_REPO_ROOT={REPO_ROOT} \
IRO_DATA_ROOT={DATA_ROOT} \
IRO_DOWNLOAD=false \
IRO_ENV_ACTIVATE={REPO_ROOT / "activate_iro_env.sh"} \
IRO_ALGORITHMS=erm,groupdro,iro \
IRO_DEBUG_DATA=0 \
IRO_STEPS=20000 \
IRO_BATCH_SIZE=32 \
IRO_N_ENVS_PER_BATCH=4 \
IRO_EVAL_SPLITS=val \
IRO_SLURM_PARTITION=gpu \
IRO_SLURM_GRES=gpu:A100:1 \
IRO_SLURM_CPUS_PER_TASK=8 \
IRO_SLURM_MEM=0 \
IRO_SLURM_TIME=23:00:00 \
IRO_EXP_NAME_PREFIX=iwildcam_long \
IRO_EXTRA_OVERRIDES="training.eval_freq=500;training.output_root={REPO_ROOT}/runs/iwildcam_long" \
./scripts/submit_iwildcam.sh
"""
print(submit_cmd)


## Collect CVaR rows and summary

This executes the generic collector and writes CSV + PNG/PDF.


In [ ]:
collect_cmd = [
    sys.executable,
    str(REPO_ROOT / "scripts/collect_cvar_curves.py"),
    str(RESULTS_ROOT),
    "--experiment", EXPERIMENT,
    "--data-root", str(DATA_ROOT),
    "--split", SPLIT,
    "--algorithms", ALGORITHMS,
    "--alpha-grid", ALPHA_GRID,
    "--output-dir", str(OUT_ROOT),
    "--device", DEVICE,
]

print("Running:")
print(" ".join(shlex.quote(part) for part in collect_cmd))
subprocess.run(collect_cmd, check=True, cwd=str(REPO_ROOT))


## Load outputs


In [ ]:
exp_out = OUT_ROOT / EXPERIMENT
summary_csv = exp_out / f"{EXPERIMENT}_cvar_summary.csv"
rows_csv = exp_out / f"{EXPERIMENT}_cvar_rows.csv"
selected_csv = exp_out / f"{EXPERIMENT}_selected_runs.csv"

assert summary_csv.exists(), f"Missing: {summary_csv}"
assert rows_csv.exists(), f"Missing: {rows_csv}"
assert selected_csv.exists(), f"Missing: {selected_csv}"

summary_df = pd.read_csv(summary_csv)
rows_df = pd.read_csv(rows_csv)
selected_df = pd.read_csv(selected_csv)

display(selected_df.sort_values(["algorithm", "seed"]))
display(summary_df.head(20))


## DGIL-style CVaR curve (SciencePlots)


In [ ]:
fig_png = exp_out / f"{EXPERIMENT}_cvar_curve_science.png"
fig_pdf = exp_out / f"{EXPERIMENT}_cvar_curve_science.pdf"

fig = plot_cvar_alpha_curves(
    summary_df,
    output_png=fig_png,
    output_pdf=fig_pdf,
    split=SPLIT,
    smooth=True,
    title=f"iWildCam CVaR curve ({SPLIT} split)",
)
plt.show()
print("Saved:", fig_png)
print("Saved:", fig_pdf)


## Table for paper-style reporting


In [ ]:
table = (
    summary_df
    .pivot(index="alpha_op", columns="algorithm", values="cvar_mean")
    .sort_index()
)
display(table)

rank_table = table.rank(axis=1, method="min")
display(rank_table)
